# NEXUS Football Intelligence — bağımsız doğrulama

**Ürünleştirme yok · Desktop Commander yok · GitHub Actions / CircleCI tetiklemesi yok · BigQuery sorgusu yok.**

Bu notebook, GitHub'dan en güncel `main` kodunu alıp ücretsiz Google Colab veya Kaggle CPU ortamında kaynak denetimi, ML regresyon testleri ve ardından isteğe bağlı tam Python testlerini çalıştırır. Sürüm kimliği, sonuçlar ve hata dökümleri JSON raporuna kaydedilir. Gerçek BigQuery veri tazeliği ve model yeniden değerlendirmesi **ayrı, yetkili bir aşamadır**; burada otomatik çalıştırılmaz.

**Kaggle:** Settings → Internet → On gerekir. **Colab:** ücretsiz CPU çalışma zamanı yeterlidir. Proje halka açık; API anahtarı veya Google Cloud kimlik bilgisi eklemeyin.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path('/kaggle/working' if Path('/kaggle/working').is_dir() else '/content' if Path('/content').is_dir() else Path.cwd())
REPO = ROOT / 'football-player-performance-analysis'
URL = 'https://github.com/senanurcetin/football-player-performance-analysis.git'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--force', 'FETCH_HEAD'], check=True)
print('Repository:', REPO)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

## 1. Gerekli paketleri yükle

Orijinal depoda sabitlenmiş paket sürümleri kullanılır. BigQuery istemci paketinin kurulması **BigQuery sorgusu çalıştırmaz** ve herhangi bir bulut kimlik bilgisi talep etmez.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '-r', str(REPO/'requirements.txt'), '-r', str(REPO/'requirements-ml.txt'),
    'PyYAML==6.0.3', 'Jinja2==3.1.6',
], check=True)
print('Dependency installation complete')

## 2. Kaynak + ML doğrulaması (Power BI ve CI yok)

Bu aşama `python -m unittest` ve kaynak analizlerini notebook CPU'sunda çalıştırır. JSON raporunda bütün alt komutların sonucu tutulur. Hata olsa bile rapor üretilir; **FAIL hiçbir zaman PASS sayılmaz**.

In [ ]:
import json
RUNNER = REPO/'scripts'/'run_offline_verification.py'
if not RUNNER.is_file():
    raise FileNotFoundError('Runner not found. Confirm that the verification notebook and runner were merged to main.')
REPORT = ROOT/'nexus_football_ml_verification.json'
proc = subprocess.run([
    sys.executable, str(RUNNER), '--repo', str(REPO),
    '--phases', 'source', 'ml', '--report', str(REPORT), '--timeout', '1200',
], check=False)
summary = json.loads(REPORT.read_text(encoding='utf-8'))
print('Result:', summary['overall_status'], '| Commit:', summary['commit_sha'])
for item in summary['results']:
    print(item['status'].upper(), item['name'])
if proc.returncode:
    print('Some checks failed. See output_tail in the report; do not claim acceptance.')

## 3. Tam depo Python testleri (yalnızca ilk aşama PASS ise)

Bu da **CI değildir**; notebook'un CPU ortamındaki yerel test çalıştırmasıdır. Power BI Desktop uygulamasını veya canlı bağlantılarını başlatmaz. Statik Power BI kaynak testleri tam ünittest kapsamına dahil olabilir; gerçek Power BI kabulü için yeterli değildir.

In [ ]:
FULL_REPORT = ROOT/'nexus_football_full_verification.json'
if summary['overall_status'] != 'PASS':
    print('SKIPPED: First verification phase is not PASS. Fix failures before continuing.')
else:
    p2 = subprocess.run([
        sys.executable, str(RUNNER), '--repo', str(REPO),
        '--phases', 'full', '--report', str(FULL_REPORT), '--timeout', '1200',
    ], check=False)
    full = json.loads(FULL_REPORT.read_text(encoding='utf-8'))
    print('Full unittest:', full['overall_status'], 'Commit:', full['commit_sha'])
    if p2.returncode:
        print('Full repository verification FAIL; inspect report, do NOT initiate final CI.')

## 4. Sonuç raporunu sakla

Çalışan notebook'u ve oluşturulan JSON raporlarını saklayın. Dosya, commit SHA ile eşlenebilir test kanıtıdır. **BigQuery'de gerçek veri, üretim ML metrikleri, Power BI Desktop ve son CI ayrıca onaylanmadan tamamlanmış sayılmaz.**

In [ ]:
from zipfile import ZipFile
archive = ROOT/'nexus_football_verification_reports.zip'
with ZipFile(archive,'w') as z:
    for path in [REPORT, FULL_REPORT]:
        if path.is_file():
            z.write(path,arcname=path.name)
print('Verification archive:', archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Kaggle/local: find the archive in notebook output files.')